In [2]:
import pandas as pd
import re 
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import cess_esp
import spacy
from nltk import CFG
import json

In [3]:
nltk.download('cess_esp')
try:
    nltk.data.find('tokenizers/punkt')
    print("El recurso 'punkt' se encuentra disponible.")
except LookupError:
    print("El recurso 'punkt' no fue encontrado. Intentando descargarlo nuevamente.")
    nltk.download('punkt')

tagger = nltk.UnigramTagger(nltk.corpus.cess_esp.tagged_sents())
nlp = spacy.load('es_core_news_sm')

[nltk_data] Downloading package cess_esp to
[nltk_data]     C:\Users\ma907\AppData\Roaming\nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


El recurso 'punkt' se encuentra disponible.


In [4]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


### Limpiando los datos para poder empezar a trabajar sobre ellos

In [5]:
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(text):
    cleaned_text = re.sub(r'/\S+', '', text)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def normalize_text(text):
    return text.lower()

def tokenizar(texto):
    doc = nlp(texto)
    return [token.text for token in doc]

def etiquetar_pos_spacy(texto):
    doc = nlp(texto)
    return [(token.text, token.pos_) for token in doc]

def lemmatize(texto):
    doc = nlp(texto)
    lemmatized_text = ' '.join([token.lemma_ for token in doc])
    
    return lemmatized_text


#### Ahora vamos a empezar a extraer features de interés y vamos empezar con el precio y la moneda en que se haría la negociación

In [6]:
def extract_price(message):
    prices = re.findall(r'\b\d{2,6}(?:[.,]\d+)? | \d{2,6}(?:[.,]\d+)? mil\b', message)
    if prices:
        return prices
    return None
def extract_currency(message):
    currencies = re.findall(r'\b(USD|usd|dólar|dolar|EURO|euro|MLC|mlc|CUP|cup|pesos|mn|dolar|dolares|mil)\b', message, re.IGNORECASE)
    if currencies:
        return currencies
    return None

##### Vamos a ir probando

In [7]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalize_text)
#data['tokens'] = data['message'].apply(tokenizar)
#data['entidades'] = data['message'].apply(extract_location)
data['precio'] = data['message'].apply(extract_price)
data['moneda'] = data['message'].apply(extract_currency) 

print(data)


                                                message     precio moneda
0                     renta x días de apto en el vedado       None   None
1     no tienes permisos para ejecutar este comando ...       None   None
2                                                             None   None
8     casa en venta en la zona sur cerca de las fábr...    [2800 ]  [usd]
9     busco renta por tiempo indefinido para una par...  [ 20 mil]  [mil]
...                                                 ...        ...    ...
4988  busco alquiler en el vedado límite 150 verde s...     [150 ]   None
4992     busco alquiler por tiempo indefinido, 58316712       None   None
4993   busco alquiler en la lisa o lo más cerca posible       None   None
4994                          busco alquiler en la lisa       None   None
4996  busco alquiler en playa, marianao, lisa hasta ...       None   None

[2508 rows x 3 columns]


##### Intentemos extraer las ubicaciones usando regex, ya que el modelo preentrenado de spacy no resultó muy útil.

In [8]:
with open("barrios_calles_habana.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

streets = [r"(calle\s+\w+(?:\s+\w+)*|calzada\s+\w+(?:\s+\w+)*)"]
intersections = r"((entre|esquina|y)\s+(calle|calzada|av\.\?|avenida)\s+\w+(?:\s+\w+)*)"
neighborhoods = [r"(centro Habana|vedado|boyeros|playa|marianao|bahía|miramar)"]
municipalities = [r"(10 de Octubre|san Miguel|habana del este|santos suárez)"]
nearby = r"(cerca de\s+\w+(?:\s+\w+)*)"
landmarks = r"(Terminal de Ómnibus Nacionales|Plaza de la Revolución|Calixto García|Pediátrico de Centro Habana|Fajardo|Ciudad Deportiva)"

for municipality, areas in json_data.items():
    municipalities.append(re.escape(municipality.lower())) 

    for neighborhood, streets_list in areas.items():
        neighborhoods.append(re.escape(neighborhood.lower())) 
        for street in streets_list:
            streets.append(re.escape(street.lower()))

streets_pattern = "|".join(streets)
neighborhoods_pattern = "|".join(neighborhoods)
municipalities_pattern = "|".join(municipalities)

location_pattern = rf"\b({streets_pattern}|{intersections}|{neighborhoods_pattern}|{municipalities_pattern}|{nearby}|{landmarks})\b"

def find_locations(message):
    matches = re.findall(location_pattern, message, re.IGNORECASE)
    unique_matches = list(set([match[0].strip() if isinstance(match, tuple) else match.strip() for match in matches]))
    return sorted(unique_matches)

data['ubicaciones'] = data['message'].apply(find_locations)

data

,message,precio,moneda,ubicaciones
0,renta x días de apto en el vedado,None,None,[el vedado]
1,no tienes permisos para ejecutar este comando ...,None,None,[]
2,,None,None,[]
8,casa en venta en la zona sur cerca de las fábr...,[2800 ],[usd],"[cerca de las fábricas de cerveza y galleta, h..."
9,busco renta por tiempo indefinido para una par...,[ 20 mil],[mil],[]
...,...,...,...,...
4988,busco alquiler en el vedado límite 150 verde s...,[150 ],None,[el vedado]
4992,"busco alquiler por tiempo indefinido, 58316712",None,None,[]
4993,busco alquiler en la lisa o lo más cerca posible,None,None,[la lisa]
4994,busco alquiler en la lisa,None,None,[la lisa]


#### Ahora vamos a extraer otro feature referente a el tipo de renta (ya sea lineal, por horas, por mes, por dia, por semanas, etc)

In [12]:
def extract_rent_duration(text):
    patterns = {
        "por días": r"(por\s\d+\sdías?|por\s24\s?horas|por\s\d+\s?días?)",
        "por hora": r"(por\s\d+\shoras?|por\s?hora|por\s?horas)",
        "indefinido": r"(por\stiempo\sindefinido|para\ssiempre)",
        "lineal": r"(al\smes|mensual|por\smes|lineal)",
        "por tiempo limitado": r"(desde\s\d{1,2}(am|pm)?\shasta\s\d{1,2}(am|pm)?|por\ssemanas?|por\stemporadas?)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    return "no especificado"

data["duration"] = data["message"].apply(extract_rent_duration)
print(data)

                                                message     precio moneda  \
0                     renta x días de apto en el vedado       None   None   
1     no tienes permisos para ejecutar este comando ...       None   None   
2                                                             None   None   
8     casa en venta en la zona sur cerca de las fábr...    [2800 ]  [usd]   
9     busco renta por tiempo indefinido para una par...  [ 20 mil]  [mil]   
...                                                 ...        ...    ...   
4988  busco alquiler en el vedado límite 150 verde s...     [150 ]   None   
4992     busco alquiler por tiempo indefinido, 58316712       None   None   
4993   busco alquiler en la lisa o lo más cerca posible       None   None   
4994                          busco alquiler en la lisa       None   None   
4996  busco alquiler en playa, marianao, lisa hasta ...       None   None   

                                            ubicaciones         duration  


In [18]:
def extraer_rent_type(text):
    patterns = {
        "apartamento": r"(apto|apartamento)",
        "casa independiente": r"(casa\sindependiente|casa\b)",
        "habitación": r"(habitación|habitación\sindependiente)",
        "estudio": r"(estudio)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    
    return "otro" 

data["tipo de renta"] = data["message"].apply(extraer_rent_type)
print(data) 


                                                message     precio moneda  \
0                     renta x días de apto en el vedado       None   None   
1     no tienes permisos para ejecutar este comando ...       None   None   
2                                                             None   None   
8     casa en venta en la zona sur cerca de las fábr...    [2800 ]  [usd]   
9     busco renta por tiempo indefinido para una par...  [ 20 mil]  [mil]   
...                                                 ...        ...    ...   
4988  busco alquiler en el vedado límite 150 verde s...     [150 ]   None   
4992     busco alquiler por tiempo indefinido, 58316712       None   None   
4993   busco alquiler en la lisa o lo más cerca posible       None   None   
4994                          busco alquiler en la lisa       None   None   
4996  busco alquiler en playa, marianao, lisa hasta ...       None   None   

                                            ubicaciones         duration  \